### contextual compression retriever 
for example, if the paragraph contains one single line which is relevant to the query an rest of the para is irrelevant , then it only gives the relevant part of it 

In [6]:
from langchain_community.vectorstores import Chroma 
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.documents import Document
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [3]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [4]:
embeddings = OllamaEmbeddings(model='nomic-embed-text')
vector_store = Chroma.from_documents(
    embedding = embeddings,
    documents = docs
)

In [5]:
base_retriever = vector_store.as_retriever(search_kwargs ={'k':5})

In [7]:
# set up compression using llm 
llm = ChatOllama(model ="qwen3:8b", reasoning= False)
compressor = LLMChainExtractor.from_llm(llm=llm)

In [8]:
# create contextual compression retriever

compression_retriever = ContextualCompressionRetriever(
    base_retriever= base_retriever,
    base_compressor= compressor
)

In [9]:
# give query to retriever 
query = "what is photosynthesis"
compression_results = compression_retriever.invoke(query)

In [10]:
for i in compression_results:
    print(i.page_content)

Photosynthesis is the process by which green plants convert sunlight into energy.
The chlorophyll in plant cells captures sunlight during photosynthesis.
Photosynthesis does not occur in animal cells.
